# Task 1 — Dataset(s) & label design (exercise type + quality)

This notebook identifies suitable datasets for workout form assessment (based on your primary source `arXiv:2202.14019`) and, importantly, defines the *real* label signals available in this project.

## What labels do we have in *this repo*?

- Exercise type: **squat** (your videos are in `videos_squat/`).
- Quality proxy: **knee-error intervals** from:
  - `error_knees_forward.json`
  - `error_knees_inward.json`

Each entry is a list of `[start_time_sec, end_time_sec]` intervals per video. We convert those intervals into frame-level quality masks during benchmarking.

## External dataset candidates (from the paper)

- **Fitness-AQA** (from `arXiv:2202.14019`): expert-trainer annotated exercise errors for multiple exercise types (BackSquat, BarbellRow, OverheadPress), i.e., exercise-type labels + quality/error labels.
- If access is possible in your environment, Fitness-AQA is a strong fit for “exercise type + quality”.

> Note: this repo focuses on using the *real* squat video data you already provided. So even if you later add Fitness-AQA, the current pipeline will still be grounded in these videos.


In [ ]:
from pathlib import Path
import numpy as np

from src.dataset_labels import load_dataset_manifest

root = Path.cwd()
manifest = load_dataset_manifest(root_dir=root, videos_dir_name="videos_squat")

print("Loaded samples:", len(manifest))
splits = {s: 0 for s in ["train", "val", "test"]}
for m in manifest:
    splits[m.split] += 1
print("Split counts:", splits)

# Quality distribution (binary proxy: any knee forward/inward interval)
quality = {"correct": 0, "incorrect": 0}
for m in manifest:
    quality["incorrect" if m.has_any_knee_error else "correct"] += 1
print("Binary quality proxy counts:", quality)

# Show a few example video IDs
correct_ids = [m.video_id for m in manifest if not m.has_any_knee_error][:10]
incorrect_ids = [m.video_id for m in manifest if m.has_any_knee_error][:10]
print("Example correct IDs:", correct_ids)
print("Example incorrect IDs:", incorrect_ids)


In [ ]:
import pandas as pd

rows = []
for m in manifest:
    rows.append(
        {
            "video_id": m.video_id,
            "split": m.split,
            "knee_forward_intervals": len(m.knee_forward_intervals),
            "knee_inward_intervals": len(m.knee_inward_intervals),
            "has_any_knee_error": m.has_any_knee_error,
        }
    )

df = pd.DataFrame(rows)
df_by_split = df.groupby("split")["has_any_knee_error"].agg(["sum", "count"])
df_by_split["incorrect_frac"] = df_by_split["sum"] / df_by_split["count"]
df_by_split = df_by_split.rename(columns={"sum": "incorrect_count", "count": "total"})

df_by_split


## Labeling summary (what we can claim in your capstone)

For this project’s data:

- **Exercise type label** exists: `squat` (all provided videos are squats).
- **Quality label** exists as a knee-error proxy:
  - `quality = 1` (incorrect) if any forward/inward knee error interval is present.
  - `quality = 0` (correct) otherwise.

This provides exactly the two-label setup your summary requires (exercise type + quality). Later you can extend to more exercises by adding additional video folders + additional error JSONs (or more general quality annotations).
